# Mini Proyecto Deep Learning - Start-Up basada en IA
#### Máster Universitario en Ciencia de Datos - UV

### Adrián Carrasco Alcalá y Clara Montalvá Barcenilla


## Descripción

El usuario dibuja su edad y un animal, añadiendo opcionalmente una pequeña descripción del mismo y del contexto deseado para la historia. Se crea tanto un dibujo mejorado del animal pensado para colorear como un cuento interactivo que lo tiene como protagonista y que va acompañado de dibujos y de una voz que narra la historia.

#### <u>Inputs</u>
- Dibujo con la edad del niño
- Dibujo de un animal
- Pequeña descripción del animal y planteamiento de la historia hecha por el usuario (opcional).

#### <u>Modelos</u>
- Red Neuronal Convolucional (CNN): Identificar la edad dibujada (entrenada con el conjunto MNIST).
- CLIP: Detectar los animales a partir de los garabatos.
- ControlNet: Mejorar los dibujos de los animales.
- VAE/ControlNet: Crear un dibujo del animal para colorear.
- API Gemini: Generar las partes de la historia.
- Suno: Generar la voz del narrador que lee la historia.
- Stable Difussion: Crear los dibujos a partir de las partes de la historia.
- ControlNet: Combinar los dibujos creados a partir de la historia con los de los animales mejorados.

#### <u>Outputs</u>
- Texto con la historia completa divida en partes, con peticiones de dibujos entre ellas.
- Audio con la voz del narrador contando la historia.
- Imágenes con dibujos creados a partir de la historia y que la complementan.
- Imagen con un dibujo mejorado del animal pintado por el usuario, preparado para colorearlo. 

### Diagrama con modelos de IA

<img src="Diagrama_Proyecto_DL.jpg" width="900" height="500">


## Aplicación de los modelos

### Red Neuronal Convolucional (CNN) para la detección de la edad

Utilizamos una red neuronal entrenada con el conjunto de datos MNIST que sea capaz de identificar el número dibujado por el niño o el padre para clasificarlo en uno de los siguientes grupos de edad:
- 3-5 años
- 6-9 años
- 10-12 años

En primer lugar, abrimos la ventana de dibujo que pregunta por la edad.

In [1]:
import numpy as np
import tensorflow as tf
import cv2
import tkinter as tk
from PIL import Image, ImageDraw
from skimage.transform import resize
from modulos.AplicacionDibujo import AplicacionDibujo

In [2]:
root = tk.Tk()  # Creamos una ventana básica en blanco
app = AplicacionDibujo(root)

# Iniciar el bucle de la ventana (el código se detiene aquí hasta que se cierre la ventana)
root.mainloop()

if app.imagen_resultado is not None:
    print("La imagen ha sido guardada.")
    img = app.imagen_resultado
else:
    print("La ventana se cerró sin guardar la imagen.")

La imagen ha sido guardada.


Cargamos el modelo.

In [3]:
modelo = tf.keras.models.load_model('modelos/CNN_edad.keras')

In [4]:
imagen = 255 - img

contornos, _ = cv2.findContours(imagen[:,:,0], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

cajas_delimitadoras = []
for c in contornos:
    x, y, w, h = cv2.boundingRect(c)
    
    if w > 10 and h > 10: # Filtramos contornos pequeños, posiblemente ruido o manchas accidentales al dibujar
        cajas_delimitadoras.append((x, y, w, h))

cajas_delimitadoras = sorted(cajas_delimitadoras, key = lambda v: v[0])

digitos = []
edad_predicha = ''

for (x, y, w, h) in cajas_delimitadoras:
        # Recortar el dígito de la imagen original
        recorte = imagen[:,:,0][y:y+h, x:x+w]

        lado_maximo = max(w, h)
        
        # Calculamos cuánto margen (padding) negro añadir a cada lado
        pad_y = (lado_maximo - h) // 2
        pad_x = (lado_maximo - w) // 2

        # Añadimos un margen extra fijo para que el número no toque los bordes (como en MNIST)
        margen_extra = int(lado_maximo * 0.3)

        recorte_cuadrado = cv2.copyMakeBorder(
                recorte, 
                pad_y + margen_extra, 
                lado_maximo - h - pad_y + margen_extra, 
                pad_x + margen_extra, 
                lado_maximo - w - pad_x + margen_extra, 
                cv2.BORDER_CONSTANT, 
                value = 0 # Queremos que el borde sea negro
        )

        imagen_28 = resize(recorte_cuadrado, (28, 28))
        imagen_res = imagen_28/imagen_28.max() 
        edad = imagen_res.reshape(1, 28, 28, 1)

        pred = modelo.predict(edad, verbose=0)

        edad_predicha += str(np.argmax(pred))

print("La edad del niño es:", edad_predicha)

La edad del niño es: 10


### Modelo Hugging Face para identificación del dibujo del animal

In [5]:
root = tk.Tk()  # Creamos una ventana básica en blanco
app = AplicacionDibujo(root, modo_edad=False)

# Iniciar el bucle de la ventana (el código se detiene aquí hasta que se cierre la ventana)
root.mainloop()

if app.imagen_resultado is not None:
    print("La imagen ha sido guardada.")
    img_animal = app.imagen_resultado
else:
    print("La ventana se cerró sin guardar la imagen.")

La imagen ha sido guardada.


In [6]:
import torch
from transformers import AutoModelForImageClassification, AutoImageProcessor

modelo_hf = "kmewhort/beit-sketch-classifier"
procesador = AutoImageProcessor.from_pretrained(modelo_hf)
modelo = AutoModelForImageClassification.from_pretrained(modelo_hf)

def clasificar_dibujo(matriz_imagen):
    # Transformamos la matriz del dibujo a una imagen PIL
    imagen = Image.fromarray(matriz_imagen.astype('uint8')).convert("RGB")
    
    # Preprocesamos la imagen
    entrada = procesador(images=imagen, return_tensors="pt")

    # Predecimos con el modelo
    with torch.no_grad():
        salida = modelo(**entrada)
        preds = salida.logits

    # Nos quedamos con el índice con la probabilidad más alta
    categoria_predicha = np.argmax(preds).item()
    
    # Dado el identificador de la categoria predicha, buscamos la etiqueta correspondiente
    animal_predicho = modelo.config.id2label[categoria_predicha]
    
    return animal_predicho

animal = clasificar_dibujo(img_animal)
print(f"El animal dibujado es: {animal}")

The image processor of type `BeitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

El animal dibujado es: whale


### API Gemini para generación de la primera parte de la historia

In [7]:
from google import genai

In [8]:
# Leemos la clave API desde un archivo de texto
with open("clave.txt", "r", encoding="utf-8") as archivo:
    GEMINI_API_KEY = archivo.read()

In [16]:
cliente = genai.Client(api_key=GEMINI_API_KEY)

modelo = "gemini-3-flash-preview"
prompt = f"Eres un experto escritor de cuentos infantiles. Escribe la primera parte de un cuento que tenga a un {animal} como protagonista de la historia. La extensión de esta primera parte debe ser de 3 párrafos. Además, el cuento debe estar escrito específicamente para que lo entienda un niño de {edad} años, utilizando un vocabulario apropiado para esa etapa de desarrollo vital. Debes tener en cuenta que el texto será leído por un narrador en voz alta, y se generará una imagen a partir de él. Debe presentarse en la historia a un segundo animal desconocido, y el texto debe acabar con una pregunta, la respuesta de la cual debería ser cuál es este nuevo animal."
respuesta = cliente.models.generate_content(
    model = modelo, contents = prompt
)

print(respuesta.text)

Había una vez una ballena muy, muy grande que se llamaba Berta. Berta era de un color azul brillante, como el cielo, y vivía en el mar profundo. Le gustaba mucho nadar despacio y lanzar chorritos de agua por su cabecita hacia arriba. ¡Pshhh, pshhh!, sonaba el agua mientras Berta jugaba muy feliz con las pompas de jabón que hacían las olas.

Un día, mientras Berta cantaba una canción bajita bajo el sol, vio a alguien nuevo cerca de unas rocas de colores. No era una ballena grande como ella, sino un animalito pequeño que nadaba muy tranquilamente. Lo más curioso de este nuevo amigo es que tenía una casita redonda y muy dura encima de su espalda, y cuatro patitas cortas que movía como si estuviera remando en un botecito.

Berta se acercó con mucho cuidado para no asustarlo y le dio un saludo muy dulce con su aleta. El animalito, muy valiente, asomó su cabecita arrugada fuera de su concha dura y movió sus ojos para mirar a la gran ballena. Este amiguito nada muy despacio por el océano y si